# 00 · Setup — DVSA Databricks connector

Installs the connector, wires Databricks Secrets, and verifies MLflow. Run this
once per cluster session. **No changes to DVSA-APIs core are required.**

In [ ]:
%pip install "git+https://github.com/ravibeta/DVSA-APIs#subdirectory=dvsa_databricks_connector"

In [ ]:
dbutils.library.restartPython()

## Create secrets (run once, from your laptop with the Databricks CLI)
```bash
databricks secrets create-scope dvsa
databricks secrets put-secret dvsa DVSA_ENDPOINT   # https://dvsa.example.com/api/v1
databricks secrets put-secret dvsa DVSA_API_KEY
```

In [ ]:
# ---- CONFIG (the only cell you must edit) --------------------------------
dbutils.widgets.text("input_delta_path", "drone.tracks", "Input Delta table/path")
dbutils.widgets.text("output_delta_path", "drone.inference_results", "Output Delta table/path")
dbutils.widgets.text("model_name", "visdrone-yolov8x", "DVSA model name")
dbutils.widgets.text("batch_size", "64", "Batch size")
dbutils.widgets.text("checkpoint_location", "/Volumes/drone/_chk", "Checkpoint (streaming)")
dbutils.widgets.text("secret_scope", "dvsa", "Databricks Secret scope")

INPUT_DELTA = dbutils.widgets.get("input_delta_path")
OUTPUT_DELTA = dbutils.widgets.get("output_delta_path")
MODEL_NAME = dbutils.widgets.get("model_name")
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
CHECKPOINT = dbutils.widgets.get("checkpoint_location")
SECRET_SCOPE = dbutils.widgets.get("secret_scope")

In [ ]:
from dvsa_databricks_connector import config_from_secrets

# Reads DVSA_ENDPOINT / DVSA_API_KEY from the secret scope. The key is never
# printed — redacted() masks it.
cfg = config_from_secrets(dbutils, scope=SECRET_SCOPE)
print(cfg.redacted())

In [ ]:
import mlflow
mlflow.set_experiment("/Shared/dvsa-drone-inference")
print("MLflow experiment ready")